In [ ]:
!pip install ipywidgets matplotlib scipy numpy -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# --- Parameters Matching Lecture Notation ---
np.random.seed(42)
N = 15000

# Noise distribution: epsilon ~ N(0, sigma^2)
mu_eps = 0.0
sigma_eps = 1.0

# Affine transformation: y = theta^T x + a * epsilon
a = 1.0
theta_T_x = 2.0  # Proxy for the deterministic prediction b

mu_y = theta_T_x + a * mu_eps
sigma_y = abs(a) * sigma_eps

# --- Sampling ---
epsilon = np.random.normal(loc=mu_eps, scale=sigma_eps, size=N)
y = theta_T_x + a * epsilon

# --- Plotting ---
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=300)

c_eps = "#1f77b4"  # Blue for zero-mean noise
c_y = "#d62728"    # Red for shifted output

grid = np.linspace(-4, 7, 1000)

# 1. Distribution of Epsilon (Noise)
ax.hist(epsilon, bins=60, density=True, alpha=0.32, color=c_eps, edgecolor="white", label=r"Noise samples $\epsilon^{(i)}$")
ax.plot(grid, norm.pdf(grid, mu_eps, sigma_eps), color=c_eps, lw=2.5,
        label=rf"$\epsilon \sim \mathcal{{N}}({int(mu_eps)},\, {int(sigma_eps)}^2)$")

# 2. Distribution of y (Target given x)
ax.hist(y, bins=60, density=True, alpha=0.32, color=c_y, edgecolor="white", label=r"Observed samples $y^{(i)}$")
ax.plot(grid, norm.pdf(grid, mu_y, sigma_y), color=c_y, lw=2.5,
        label=rf"$y \mid \boldsymbol{{x}} \sim \mathcal{{N}}(\boldsymbol{{\theta}}^T\boldsymbol{{x}},\, \sigma^2) = \mathcal{{N}}({int(mu_y)},\, {int(sigma_y)}^2)$")

# Vertical mean lines
ax.axvline(mu_eps, color=c_eps, linestyle="--", lw=1.5, alpha=0.8)
ax.axvline(mu_y, color=c_y, linestyle="--", lw=1.5, alpha=0.8)

# Arrow showing the deterministic shift
ax.annotate(
    "", xy=(mu_y, 0.35), xytext=(mu_eps, 0.35),
    arrowprops=dict(arrowstyle="->", color="black", lw=1.8)
)
ax.text((mu_eps + mu_y) / 2, 0.37, rf"Shift by deterministic mean $\boldsymbol{{\theta}}^T\boldsymbol{{x}} = {theta_T_x}$",
        ha="center", va="bottom", fontsize=11, fontweight="bold")

# Formatting
ax.set_xlabel("Value", fontsize=11)
ax.set_ylabel("Probability Density", fontsize=11)
ax.set_xlim(-4, 7)
ax.set_ylim(0, 0.48)
ax.legend(frameon=True, facecolor="white", edgecolor="none", fontsize=10, loc="upper right")

plt.tight_layout()
plt.savefig("gaussian_noise_shift.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import ipywidgets as widgets
from ipywidgets import interact

def plot_regression_noise(mu_eps=0.0, sigma_eps=1.0, a=1.0, theta_T_x=2.0):
    np.random.seed(42)
    epsilon = np.random.normal(loc=mu_eps, scale=sigma_eps, size=12000)
    y = theta_T_x + a * epsilon

    mu_y = theta_T_x + a * mu_eps
    sigma_y = abs(a) * sigma_eps

    fig, ax = plt.subplots(figsize=(9, 4.2), dpi=120)
    grid = np.linspace(-6, 10, 1000)

    # Plot Noise
    ax.hist(epsilon, bins=55, density=True, alpha=0.35, color="#1f77b4", label=r"$\epsilon \sim \mathcal{N}(\mu_\epsilon, \sigma^2)$")
    ax.plot(grid, norm.pdf(grid, mu_eps, sigma_eps), color="#1f77b4", lw=2)

    # Plot y
    ax.hist(y, bins=55, density=True, alpha=0.35, color="#d62728", label=r"$y = \boldsymbol{\theta}^T\boldsymbol{x} + a\epsilon$")
    ax.plot(grid, norm.pdf(grid, mu_y, sigma_y), color="#d62728", lw=2)

    # Annotate means
    ax.axvline(mu_eps, color="#1f77b4", linestyle="--", alpha=0.7)
    ax.axvline(mu_y, color="#d62728", linestyle="--", alpha=0.7)

    ax.set_xlim(-6, 10)
    ax.set_ylim(0, 0.55)
    ax.set_title(f"Target Distribution: Mean = {mu_y:.2f} (shifted by θ^T x = {theta_T_x:.2f}), Std Dev = {sigma_y:.2f}")
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    ax.legend(loc="upper right")
    plt.show()

interact(plot_regression_noise,
         mu_eps=widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.5, description="μ (noise):"),
         sigma_eps=widgets.FloatSlider(value=1.0, min=0.2, max=2.5, step=0.1, description="σ (noise):"),
         a=widgets.FloatSlider(value=1.0, min=-2.0, max=3.0, step=0.5, description="scale (a):"),
         theta_T_x=widgets.FloatSlider(value=2.0, min=-4.0, max=6.0, step=0.5, description="θᵀx (shift):"));